# Amharic Sentiment Analysis with Prompt Tuning (PEFT)
**A low-resource language project using data augmentation and prompt tuning**

### Project Overview
- Dataset: Amharic sentiment tweets (~2.8k samples)
- Base model: `rasyosef/bert-medium-amharic`
- Technique: Prompt Tuning (PEFT) + simple data augmentation
- Goal: Binary sentiment classification (positive / negative)
- This notebook serves as the **initial version** — future iterations will include:
  - Better augmentation (back-translation, synonyms)
  - LoRA instead of prompt tuning
  - Class weighting
  - Hyperparameter tuning
  - Model merging & deployment

Run this notebook in Google Colab with **T4 GPU** runtime for best performance.

Last updated: March 2025

In [1]:
# ────────────────────────────────────────────────
# Install required packages
# ────────────────────────────────────────────────
!pip install -q transformers datasets evaluate peft accelerate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.4 MB/s eta 0:00:00


In [2]:
# ────────────────────────────────────────────────
# Import libraries
# ────────────────────────────────────────────────
import torch
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    pipeline
)
from peft import get_peft_model, PromptTuningConfig, TaskType
import evaluate
from sklearn.model_selection import train_test_split

## 1. Load and Prepare the Dataset

In [3]:
# ────────────────────────────────────────────────
# Load the Amharic sentiment dataset
# ────────────────────────────────────────────────
dataset = load_dataset("rasyosef/amharic-sentiment")
df = dataset["train"].to_pandas()

print("Dataset size:", len(df))
print("Label distribution:\n", df["label"].value_counts(normalize=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/269k [00:00<?, ?B/s]

data/dev-00000-of-00001.parquet:   0%|          | 0.00/35.2k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/35.2k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2223 [00:00<?, ? examples/s]

Generating dev split:   0%|          | 0/279 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/279 [00:00<?, ? examples/s]

Dataset size: 2223
Label distribution:
 label
0    0.536212
1    0.463788
Name: proportion, dtype: float64


In [4]:
# ────────────────────────────────────────────────
# Stratified train/test split (80/20)
# ────────────────────────────────────────────────
train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset  = Dataset.from_pandas(test_df.reset_index(drop=True))

print(f"Train: {len(train_df):,} | Test: {len(test_df):,}")

Train: 1,778 | Test: 445


## 2. Data Augmentation (basic version)
Simple adjacent word swapping to increase training data size

In [5]:
# ────────────────────────────────────────────────
# Simple augmentation function
# ────────────────────────────────────────────────
def simple_augment(text, num_aug=2):
    words = text.split()
    augmented = []
    for _ in range(num_aug):
        if len(words) > 3:
            idx = np.random.randint(0, len(words)-1)
            words[idx], words[(idx+1) % len(words)] = words[(idx+1) % len(words)], words[idx]
        augmented.append(" ".join(words))
    return augmented

# Generate augmented examples
aug_texts, aug_labels = [], []
for text, label in zip(train_df["clean_tweet"], train_df["label"]):
    augs = simple_augment(text)
    aug_texts.extend(augs)
    aug_labels.extend([label] * len(augs))

# Combine original + augmented data
aug_dataset = Dataset.from_dict({
    "clean_tweet": list(train_df["clean_tweet"]) + aug_texts,
    "label": list(train_df["label"]) + aug_labels
})

print(f"Augmented training set size: {len(aug_dataset):,} samples")

Augmented training set size: 5,334 samples


## 3. Quick inference with the original fine-tuned model
(For reference — reported ~0.83 macro F1)

In [ ]:
# ────────────────────────────────────────────────
# Load and test the original fine-tuned model
# ────────────────────────────────────────────────
original_classifier = pipeline(
    "text-classification",
    model="rasyosef/bert-medium-amharic-finetuned-sentiment"
)

test_sentences = [
    "አሪፍ ፊልም ነው።",
    "በጣም መጥፎ ነው",
    "ደስ ይላል"
]

for sent in test_sentences:
    result = original_classifier(sent)[0]
    print(f"{sent:25} → {result['label']} ({result['score']:.4f})")

config.json:   0%|          | 0.00/821 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/162M [00:00<?, ?B/s]

## 4. Prompt Tuning Setup (PEFT)

In [ ]:
──────────────────────────────────────
# Load base model and tokenizer
# ────────────────────────────────────────────────
model_name = "rasyosef/bert-medium-amharic"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
Python

In [ ]:
# ────────────────────────────────────────────────
# Configure Prompt Tuning
# ────────────────────────────────────────────────
peft_config = PromptTuningConfig(
    task_type=TaskType.SEQ_CLS,
    num_virtual_tokens=20,
    prompt_tuning_init="TEXT",
    prompt_tuning_init_text="Classify the Amharic tweet as positive or negative: ",
    tokenizer_name_or_path=model_name,
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

## 5. Tokenization

In [ ]:
# ────────────────────────────────────────────────
# Tokenize datasets
# ────────────────────────────────────────────────
def tokenize_function(examples):
    return tokenizer(
        examples["clean_tweet"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized_train = aug_dataset.map(tokenize_function, batched=True)
tokenized_test  = test_dataset.map(tokenize_function, batched=True)

## 6. Training Setup

In [ ]:
# ────────────────────────────────────────────────
# Training arguments
# ────────────────────────────────────────────────
training_args = TrainingArguments(
    output_dir="./amharic-sentiment-results",
    eval_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    save_strategy="no",
    report_to="none",
    logging_steps=50,
)

In [ ]:
# ────────────────────────────────────────────────
# Evaluation metrics
# ────────────────────────────────────────────────
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    f1 = evaluate.load("f1").compute(
        predictions=predictions,
        references=labels,
        average="macro"
    )
    return {"macro_f1": f1["f1"]}

In [ ]:
# ────────────────────────────────────────────────
# Initialize Trainer and train
# ────────────────────────────────────────────────
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    compute_metrics=compute_metrics,
)

trainer.train()

## 7. Final Evaluation

In [ ]:
# ────────────────────────────────────────────────
# Evaluate on test set
# ────────────────────────────────────────────────
results = trainer.evaluate()
print("Final evaluation results:")
print(results)
Markdown

## 8. Quick Manual Inference with the PEFT model

In [ ]:
# ────────────────────────────────────────────────
# Manual inference test
# ────────────────────────────────────────────────
test_examples = [
    "አሪፍ ፊልም ነው።",
    "በጣም ጥሩ ነው ይህ",
    "በጣም መጥፎ ነው",
    "ደስ ይላል",
]

for text in test_examples:
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True).to(model.device)
    with torch.no_grad():
        outputs = model(**inputs)
    logits = outputs.logits
    pred = torch.argmax(logits, dim=-1).item()
    prob = torch.softmax(logits, dim=-1)[0][pred].item()
    label_str = "Positive" if pred == 1 else "Negative"
    print(f"{text:30} → {label_str} ({prob:.4f})")

## Next Steps (planned improvements)
- Add class weighting to handle imbalance
- Use better augmentation (back-translation with NLLB)
- Switch to LoRA for better stability and performance
- Save PEFT adapter properly
- Add confusion matrix & per-class metrics
- Create Gradio demo